# PyGraphDB Property Graph API

This notebook demonstrates the current PyGraphDB API for working with a small attributed property graph. Nodes and edges store arbitrary property dictionaries, while adjacency records support graph traversal.

## Local Development Import

When running this notebook from the repository checkout, add `../src` to `sys.path`. If the package is already installed, this cell is harmless.

In [ ]:
from pathlib import Path
import sys

repo_src = Path.cwd().parent / 'src'
if repo_src.exists():
    sys.path.insert(0, str(repo_src))

In [ ]:
import shutil
import tempfile

from pygraphdb.graphdb import Edge, GraphDB, Node
from pygraphdb.kvstores import LMDBStore
from pygraphdb.serializers import PickleSerializer

## Create a Graph

Use a key-value store backend and a serializer. This notebook uses a temporary LMDB directory so it leaves no persistent files behind.

In [ ]:
db_path = tempfile.mkdtemp(prefix='pygraphdb_property_graph_')
store = LMDBStore(path=db_path)
graph = GraphDB(store, PickleSerializer())
db_path

## Insert Nodes

Nodes have an ID and a `properties` dictionary. If no ID is supplied, PyGraphDB generates a UUID.

In [ ]:
alice = Node(node_id='alice', properties={'kind': 'person', 'name': 'Alice', 'age': 30})
bob = Node(node_id='bob', properties={'kind': 'person', 'name': 'Bob', 'age': 34})
carol = Node(node_id='carol', properties={'kind': 'person', 'name': 'Carol', 'age': 29})
acme = Node(node_id='acme', properties={'kind': 'company', 'name': 'Acme Corp'})

for node in [alice, bob, carol, acme]:
    graph.put_node(node)

[graph.get_node(node.get_id_bytes).to_dict() for node in [alice, bob, carol, acme]]

## Insert Edges

Edges have an ID, source node ID, target node ID, and their own `properties` dictionary. A relationship type can be represented as a property such as `relation`.

In [ ]:
edges = [
    Edge(edge_id='alice-knows-bob', source='alice', target='bob', properties={'relation': 'KNOWS', 'since': 2020}),
    Edge(edge_id='bob-knows-carol', source='bob', target='carol', properties={'relation': 'KNOWS', 'since': 2021}),
    Edge(edge_id='alice-works-at-acme', source='alice', target='acme', properties={'relation': 'WORKS_AT', 'role': 'Engineer'}),
]

graph.put_edges_bulk(edges)

[graph.get_edge(edge.get_id_bytes).to_dict() for edge in edges]

## Read Adjacency Lists

Adjacency lists are maintained when edges are inserted. Use `direction='forward'`, `direction='backward'`, or `direction='any'` to inspect incident edge IDs.

In [ ]:
{
    'alice_forward': graph.get_adjacency_list(b'alice', direction='forward'),
    'alice_backward': graph.get_adjacency_list(b'alice', direction='backward'),
    'alice_any': graph.get_adjacency_list(b'alice', direction='any'),
    'bob_any': graph.get_adjacency_list(b'bob', direction='any'),
}

## Traverse with BFS

`bfs` returns node IDs as bytes. The default `direction='any'` treats edges as undirected for traversal.

In [ ]:
edge_key_serializer = lambda edge_id: edge_id
[node_id.decode('utf-8') for node_id in graph.bfs(b'alice', direction='any', edge_key_serializer=edge_key_serializer)]

## Query by Properties

The current API stores properties but does not expose a high-level property index. For small result sets or examples, iterate over keys and filter deserialized nodes or edges in Python.

In [ ]:
def all_nodes(graph):
    for node_key in graph.get_node_keys_generator():
        yield graph.get_node(node_key)

people = [node.to_dict() for node in all_nodes(graph) if node.properties.get('kind') == 'person']
people_over_30 = [node.to_dict() for node in all_nodes(graph) if node.properties.get('age', 0) > 30]

{
    'people': people,
    'people_over_30': people_over_30,
}

## Update Properties

`update_node` and `update_edge` accept a merge function. The function receives the old properties and the new data, then returns the merged property dictionary.

In [ ]:
def merge_properties(old_properties, new_properties):
    merged = dict(old_properties)
    merged.update(new_properties)
    return merged

graph.update_node(b'alice', {'department': 'Engineering'}, merge_properties)
graph.update_edge(b'alice-knows-bob', {'strength': 'high'}, merge_properties)

{
    'alice': graph.get_node(b'alice').to_dict(),
    'alice_knows_bob': graph.get_edge(b'alice-knows-bob').to_dict(),
}

## Delete Records

Nodes and edges can be deleted by byte key. Deleting an edge removes the edge record and attempts to remove it from adjacency lists.

In [ ]:
graph.delete_edge(b'bob-knows-carol')
graph.delete_node(b'carol')

{
    'carol': graph.get_node(b'carol'),
    'bob_knows_carol': graph.get_edge(b'bob-knows-carol'),
    'bob_adjacency': graph.get_adjacency_list(b'bob', direction='any'),
}

## Close and Clean Up

In [ ]:
graph.close()
shutil.rmtree(db_path, ignore_errors=True)